In [1]:
# set parameters
from glob import glob

params = {
    'csv_files': glob('../data/registered_points/*all*.csv'), #csv files in raw image space to be transformed
    'mesh_path': '../code/ccf_files/CCF_meshes/json_verts_float', #location of ccf meshes for plotting
    'ccf_res': 25,
}

In [2]:
# Functions for loading mesh for viewing
import os
import json
import pickle

import numpy as np

def get_region_lists():
    """
    Import list of acronyms of brain regions
    """
    
    CCF_dir = '../code/ccf_files/CCF_meshes'
    
    
    # Reading non-crossing structures to get acronyms
    with open(os.path.join(CCF_dir, "non_crossing_structures"), "rb") as f:
        hemi_struct = pickle.load(f)
        hemi_struct.remove(1051)  # don't know why this is being done
        hemi_labeled = [(s, "hemi") for s in hemi_struct]

    # Reading mid-crossing structures to get acronyms
    with open(os.path.join(CCF_dir, "mid_crossing_structures"), "rb") as f:
        u = pickle._Unpickler(f)
        u.encoding = "latin1"
        mid_struct = u.load()
        mid_labeled = [(s, "mid") for s in mid_struct]

    return hemi_labeled + mid_labeled

def load_json_mesh(root, struct):
    """
    Loads meshes that are stored in json files like the CCF meshes

    Parameters
    ----------
    root : str
        path to mesh
    struct: str
        the region id for mesh you want to load

    Returns
    -------
    tuple
        The vertices and faces of a given mesh

    """
        
    region_metadata = get_region_lists()
    region_dict = {i[0]: i[1] for i in region_metadata}
    fname = os.path.join(root, f"{struct}.json")
        
    with open(fname) as f:
        structure_data = json.loads(f.read())
        verts, faces = (
            np.array(structure_data[struct]["vertices"]),
            np.array(structure_data[struct]["faces"]),
        )
            
        if region_dict[int(struct)] == 'hemi':
            offset = faces.max() + 1
            faces = np.vstack((faces, faces + offset))
                
            verts_2 = verts.copy()
            verts_2[:, 0] = verts_2[:, 0] + (5700 - verts_2[:, 0]) * 2
            verts = np.vstack((verts, verts_2))
            
    return verts, faces

In [3]:
mesh_ids = [771, 997]

mesh_dict = {}

for mesh_id in mesh_ids:
    verts, faces = load_json_mesh(params['mesh_path'], str(mesh_id))
    mesh_dict[str(mesh_id)] = {
        "verts": verts,
        "faces": faces
    }
    

    

In [4]:
# make mesh watertight: 
import k3d
import matplotlib as mpl
import point_cloud_utils as pcu

def rgb_to_hex(r,g,b):
    # Convert to a hexadecimal string
    hex_color = f'{r:02x}{g:02x}{b:02x}'
    # Convert the hexadecimal string to an integer in base-16
    color_int = int(hex_color, 16)
    return color_int


mesh_id = '771'

v, f = mesh_dict[str(mesh_id)]['verts'], mesh_dict[str(mesh_id)]['faces']
plot = k3d.plot()



mesh = k3d.mesh(v, f, color=rgb_to_hex(255,0,255), opacity=0.3)
plot += mesh

resolution = 5_000
vw, fw = pcu.make_mesh_watertight(v, f, resolution)

mesh = k3d.mesh(vw, fw, color=rgb_to_hex(0,255, 255), opacity=0.3)
plot += mesh

resolution = 1_000
vw, fw = pcu.make_mesh_watertight(v, f, resolution)

mesh = k3d.mesh(vw, fw, color=rgb_to_hex(255, 255, 0), opacity=0.3)
plot += mesh

plot.display()



/opt/conda/lib/python3.9/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "float64" does not match required type "float32". A coerced copy has been created.
  warnings.warn(
/opt/conda/lib/python3.9/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "int64" does not match required type "uint32". A coerced copy has been created.
  warnings.warn(


Output()

In [5]:
# get registered points
import pandas as pd

point_dict = {}
for file in params['csv_files']:
    lt_id = os.path.basename(file).split('_')[0]
    df = pd.read_csv(file, index_col = 0)
    points = df.values
    point_dict[lt_id] = points[:, [2, 1, 0]] * params['ccf_res']
    
print(point_dict.keys())

mesh_id = 997
dataset = '751017'
v, f = mesh_dict[str(mesh_id)]['verts'], mesh_dict[str(mesh_id)]['faces']
plot = k3d.plot()

mesh = k3d.mesh(v, f, color=rgb_to_hex(128, 128, 128), opacity=0.05)
plot += mesh

pts = k3d.points(positions = point_dict[dataset].astype('float32'), point_size = 50, color = rgb_to_hex(0, 0, 0))
plot += pts

plot.display()

dict_keys(['755073', '751019', '751017', '755072', '755069', '762199', '762196'])


/opt/conda/lib/python3.9/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "float64" does not match required type "float32". A coerced copy has been created.
  warnings.warn(
/opt/conda/lib/python3.9/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "int64" does not match required type "uint32". A coerced copy has been created.
  warnings.warn(


Output()

In [6]:
query_pts = point_dict[dataset]

print(query_pts.shape)

mesh_id = '771'
v, f = mesh_dict[str(mesh_id)]['verts'], mesh_dict[str(mesh_id)]['faces']

v = np.array(v, order = 'F')

print(v.shape)

sdf, fid, bc = pcu.signed_distance_to_mesh(query_pts, v, f)

print(sdf)

(2687, 3)
(117916, 3)
[5880.84686623 2668.00897738 2556.90737746 ... -111.4304698   -57.11486823
  -68.30411045]


In [8]:
inside = []
boarder = []
outside = []

thresh = 300

for c, s in enumerate(sdf):
    if s < 0:
        inside.append([query_pts[c, :]])
    elif s < thresh:
        boarder.append([query_pts[c, :]])
    else:
        outside.append([query_pts[c, :]])

plot = k3d.plot()        
        
v, f = mesh_dict[str(997)]['verts'], mesh_dict[str(997)]['faces']
mesh = k3d.mesh(v, f, color=rgb_to_hex(128, 128, 128), opacity=0.05)
plot += mesh
        
        
v, f = mesh_dict[str(771)]['verts'], mesh_dict[str(771)]['faces']
mesh = k3d.mesh(v, f, color=rgb_to_hex(0, 255, 255), opacity=0.05)
plot += mesh

pts = k3d.points(positions = np.array(inside).astype('float32'), point_size = 50, color = rgb_to_hex(255, 0, 0))
plot += pts

pts = k3d.points(positions = np.array(boarder).astype('float32'), point_size = 50, color = rgb_to_hex(0, 255, 0))
plot += pts

pts = k3d.points(positions = np.array(outside).astype('float32'), point_size = 50, color = rgb_to_hex(0, 0, 0))
plot += pts

plot.display()

Output()